# Dynamic-Fee-AMM — Phase 4 Backtest

**Twin-engine simulation proving that the dynamic volatility fee outperforms a static 0.3% fee through a simulated 24-hour market crash and recovery.**

Each minute the pool is re-pegged to the exogenous market price by an arbitrage trade — large during the crash, tiny when calm. Model A charges a flat 30 bps on that flow; Model B charges the Phase-3 volatility-scaled dynamic fee (30–150 bps). Retained fees compound into the constant product `k`, so the pool that captures more fee ends the day worth more.

| | Model A | Model B |
|---|---|---|
| Fee model | Static 30 bps | Dynamic 30–150 bps |
| Initial ETH reserve | 100 | 100 |
| Initial USDC reserve | 300,000 | 300,000 |

In [ ]:
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Output directory for chart PNGs
PLOTS_DIR = 'plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f'Charts will be saved to: {os.path.abspath(PLOTS_DIR)}')

## 1. Protocol Constants

In [ ]:
# --- Pool initialisation ---
X0            = 100.0       # initial ETH reserve
Y0            = 300_000.0   # initial USDC reserve
INITIAL_PRICE = Y0 / X0    # 3,000 USDC/ETH
INITIAL_VALUE = 2.0 * Y0   # 600,000 USDC  (pool value = 2*y at marginal price)

# --- Phase 3 fee constants (exact match to Solidity) ---
BASE_FEE_BPS     = 30        # 0.30% floor
MAX_FEE_BPS      = 150       # 1.50% ceiling
DECAY_HALFLIFE_S = 60        # EMA half-life in seconds
VOL_ALPHA        = 15        # fee sensitivity numerator
VOL_SCALE        = 1_000     # fee sensitivity denominator

# --- Chart palette ---
NAVY    = '#1B2A4A'          # Model A
EMERALD = '#00A878'          # Model B
GOLD    = '#F5A623'          # price / HODL
GREY    = '#8C9EAD'

print(f'Initial pool value : ${INITIAL_VALUE:,.0f} USDC')
print(f'Initial price      : ${INITIAL_PRICE:,.0f} USDC/ETH')

## 2. Market Data Generator

1,440 one-minute rows across three regimes: equilibrium → crash (−30% directed) → partial V-recovery. `delta_t` (the gap between trades) drops near zero during the crash to mimic HFT density — this is what drives the dynamic fee's volatility accumulator.

In [ ]:
def generate_market_data(seed=42):
    '''
    Synthetic 24-hour market dataset.
    Returns DataFrame: minute, price, amount_in_eth, delta_t, direction, regime
    '''
    rng = np.random.default_rng(seed)
    N   = 1440

    # ---- Price series ----
    prices = np.empty(N)
    p = 3_000.0

    # Regime 1 — equilibrium: tiny random walk
    for i in range(480):
        p *= math.exp(rng.normal(0.0, 0.001))
        prices[i] = p

    # Regime 2 — crash: directed drift toward -30% with high noise
    crash_drift = math.log(0.70) / 480
    for i in range(480, 960):
        p *= math.exp(crash_drift + rng.normal(0.0, 0.008))
        p  = max(p, 50.0)
        prices[i] = p

    # Regime 3 — recovery: sustained upward drift clawing back most of the crash;
    # lands the day roughly 10-15% below the open (partial V-shape)
    for i in range(960, 1440):
        p *= math.exp(rng.normal(0.0009, 0.005))
        prices[i] = p

    # ---- Trade sizes (ETH-equivalent), kept for reference / extensibility ----
    amount_in_eth = np.concatenate([
        rng.uniform(0.1, 1.0, 480),    # equilibrium: small retail
        rng.uniform(0.5, 5.0, 480),    # crash: large arb / MEV
        rng.uniform(0.2, 2.5, 480),    # recovery: medium
    ])

    # ---- Inter-trade time gap (seconds) — drives the dynamic-fee decay ----
    delta_t = np.concatenate([
        rng.uniform(30.0, 120.0, 480), # equilibrium: relaxed (often > 60s half-life)
        rng.uniform( 1.0,  15.0, 480), # crash: HFT density (gaps << 60s → no decay)
        rng.uniform(20.0,  90.0, 480), # recovery: moderate
    ])
    delta_t[0] = 60.0  # safe default for first row

    # ---- Direction column (informational; price is driven by arbitrage) ----
    direction = np.concatenate([
        rng.choice([1, -1], 480, p=[0.50, 0.50]),
        rng.choice([1, -1], 480, p=[0.75, 0.25]),
        rng.choice([1, -1], 480, p=[0.35, 0.65]),
    ])

    regimes = ['equilibrium']*480 + ['crash']*480 + ['recovery']*480

    return pd.DataFrame({
        'minute'        : np.arange(N),
        'price'         : prices,
        'amount_in_eth' : amount_in_eth,
        'delta_t'       : delta_t,
        'direction'     : direction,
        'regime'        : regimes,
    })


df = generate_market_data(seed=42)
print(f'Rows generated : {len(df)}')
print(f'Price open      : ${df.price.iloc[0]:,.0f}')
print(f'Crash low       : ${df.price[480:960].min():,.0f}')
print(f'Day close       : ${df.price.iloc[-1]:,.0f}  ({(df.price.iloc[-1]/3000-1)*100:+.0f}%)')
df.head(3)

## 3. AMM Simulator Class

Both engines use the same constant-product swap formula and the same arbitrage re-peg mechanism; only the fee multiplier differs. The dynamic fee logic is an exact Python replica of the Phase-3 Solidity `calculateDynamicFee`, including the **integer-floor time decay** — gaps shorter than one 60s half-life produce `shift = 0`, so during the crash the volatility accumulator compounds with no decay and the fee races to the 150 bps cap.

In [ ]:
class AMMSimulator:
    '''
    Twin-engine AMM backtest.
    Model A: static 30 bps fee.
    Model B: dynamic 30-150 bps fee (Phase 3 Solidity logic translated to Python).
    '''

    def __init__(self):
        self.a_x, self.a_y, self.a_cum_fees_usdc = X0, Y0, 0.0
        self.b_x, self.b_y, self.b_cum_fees_usdc = X0, Y0, 0.0
        self.b_vol_tracker = 0.0
        self.b_last_ts     = 0.0

    @staticmethod
    def _calc_amount_out(amount_in, reserve_in, reserve_out, fee_bps):
        '''Solidity getAmountOut replica.'''
        fee_mul = 10_000.0 - fee_bps
        num = reserve_out * amount_in * fee_mul
        den = reserve_in  * 10_000.0  + amount_in * fee_mul
        return num / den

    def _calc_dynamic_fee(self, eth_equiv_size, reserve_eth, current_ts):
        '''
        Solidity calculateDynamicFee replica.
        Integer-floor shift (timeElapsed // 60) matches the on-chain >> operator,
        so sub-half-life gaps cause zero decay and the accumulator compounds.
        '''
        time_elapsed = current_ts - self.b_last_ts
        shift = int(time_elapsed // DECAY_HALFLIFE_S)
        decayed_vol  = 0.0 if shift >= 112 else self.b_vol_tracker / (2 ** shift)
        price_impact = (eth_equiv_size * 10_000.0) / reserve_eth
        new_vol      = decayed_vol + price_impact
        raw_fee      = float(BASE_FEE_BPS) + (new_vol * VOL_ALPHA / VOL_SCALE)
        fee_bps      = max(float(BASE_FEE_BPS), min(float(MAX_FEE_BPS), raw_fee))
        return fee_bps, new_vol

    @staticmethod
    def _arb_size(x, y, target_price):
        '''
        Arbitrage trade that re-pegs the pool to the market price.
        Post-arb reserves satisfying y'/x' = target_price are
            x' = sqrt(k / target_price),  y' = sqrt(k * target_price).
        Returns (direction, amount_in_native, eth_equiv_size).
        '''
        k = x * y
        x_target = math.sqrt(k / target_price)
        if x_target >= x:
            amount_in_eth = x_target - x          # sell ETH into the pool
            return 1, amount_in_eth, amount_in_eth
        else:
            y_target       = math.sqrt(k * target_price)
            amount_in_usdc = y_target - y          # buy ETH out of the pool
            eth_equiv      = amount_in_usdc / target_price
            return -1, amount_in_usdc, eth_equiv

    def _execute_swap(self, x, y, amount_in_native, direction, fee_bps):
        '''
        direction =  1: native is ETH  (ETH in, USDC out)
        direction = -1: native is USDC (USDC in, ETH out)
        Full input (fee included) stays in the pool, so k grows by the fee.
        '''
        if x <= 1e-9 or y <= 1e-9:
            return x, y, 0.0
        pool_price = y / x
        if direction == 1:
            amount_out = self._calc_amount_out(amount_in_native, x, y, fee_bps)
            fee_usdc   = amount_in_native * (fee_bps / 10_000.0) * pool_price
            new_x, new_y = x + amount_in_native, y - amount_out
        else:
            amount_out = self._calc_amount_out(amount_in_native, y, x, fee_bps)
            fee_usdc   = amount_in_native * (fee_bps / 10_000.0)
            new_x, new_y = x - amount_out, y + amount_in_native
        if new_x <= 1e-9 or new_y <= 1e-9:
            return x, y, 0.0
        return new_x, new_y, fee_usdc

    def run(self, df):
        '''Loop over all minutes and return two per-step metric DataFrames.'''
        records_a, records_b = [], []
        ts = 0.0   # cumulative simulated time (sum of delta_t)

        for i, row in df.iterrows():
            ts          += float(row['delta_t'])
            market_price = float(row['price'])

            # --- Model A (static 30 bps) ---
            dir_a, amt_a, _ = self._arb_size(self.a_x, self.a_y, market_price)
            self.a_x, self.a_y, fee_a = self._execute_swap(
                self.a_x, self.a_y, amt_a, dir_a, float(BASE_FEE_BPS)
            )
            self.a_cum_fees_usdc += fee_a

            # --- Model B (dynamic fee) ---
            dir_b, amt_b, eth_equiv = self._arb_size(self.b_x, self.b_y, market_price)
            b_fee_bps, b_new_vol = self._calc_dynamic_fee(eth_equiv, self.b_x, ts)
            self.b_x, self.b_y, fee_b = self._execute_swap(
                self.b_x, self.b_y, amt_b, dir_b, b_fee_bps
            )
            self.b_cum_fees_usdc += fee_b
            self.b_vol_tracker    = b_new_vol
            self.b_last_ts        = ts

            # --- Metrics helper ---
            hodl_value = X0 * market_price + Y0

            def _record(px, py, cum_fees, fee_used):
                pool_price    = py / px
                pool_value    = px * market_price + py        # mark to market
                P             = pool_price / INITIAL_PRICE
                il_pct        = 2.0 * math.sqrt(P) / (1.0 + P) - 1.0
                il_usdc       = il_pct * hodl_value
                return {
                    'minute'             : i,
                    'regime'             : row['regime'],
                    'pool_x'             : px,
                    'pool_y'             : py,
                    'pool_price'         : pool_price,
                    'pool_value_usdc'    : pool_value,
                    'hodl_value_usdc'    : hodl_value,
                    'cum_fees_usdc'      : cum_fees,
                    'fee_bps'            : fee_used,
                    'il_pct'             : il_pct,
                    'il_usdc'            : il_usdc,
                    'net_lp_return_usdc' : cum_fees + il_usdc,
                }

            records_a.append(_record(
                self.a_x, self.a_y, self.a_cum_fees_usdc, float(BASE_FEE_BPS)
            ))
            records_b.append(_record(
                self.b_x, self.b_y, self.b_cum_fees_usdc, b_fee_bps
            ))

        return pd.DataFrame(records_a), pd.DataFrame(records_b)

## 4. Run the Simulation

In [ ]:
sim = AMMSimulator()
df_a, df_b = sim.run(df)

print(f'Simulation complete — {len(df_a)} records per model')
print(f'\nPool tracks market : A=${df_a.pool_price.iloc[-1]:,.0f}  B=${df_b.pool_price.iloc[-1]:,.0f}  market=${df.price.iloc[-1]:,.0f}')
print(f'\nDynamic fee by regime (Model B):')
for r in ['equilibrium', 'crash', 'recovery']:
    sub = df_b[df_b.regime == r]
    print(f'  {r:<12}: avg {sub.fee_bps.mean():5.1f} bps   max {sub.fee_bps.max():5.1f} bps')
print(f'\nModel A cum fees : ${df_a.cum_fees_usdc.iloc[-1]:>10,.2f} USDC')
print(f'Model B cum fees : ${df_b.cum_fees_usdc.iloc[-1]:>10,.2f} USDC')

## 5. Chart 1 — Fee Elasticity vs. Market Volatility

The dynamic fee spikes from the 30 bps floor toward the 150 bps ceiling during the crash regime (high-frequency arbitrage flow, no time decay), then bleeds back to the floor as the recovery's longer gaps let the accumulator decay.

In [ ]:
def _shade(ax):
    ax.axvspan(480,  960, color='red',   alpha=0.10, zorder=0, label='Crash Regime')
    ax.axvspan(960, 1440, color='green', alpha=0.06, zorder=0, label='Recovery Regime')

sns.set_style('darkgrid')
fig, ax1 = plt.subplots(figsize=(14, 6))
_shade(ax1)

ax1.plot(df['minute'], df['price'], color=GOLD, lw=1.5, label='ETH Market Price')
ax1.set_xlabel('Time (minutes)', fontsize=12)
ax1.set_ylabel('ETH Price (USDC)', color=GOLD, fontsize=12)
ax1.tick_params(axis='y', labelcolor=GOLD)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

ax2 = ax1.twinx()
ax2.plot(df_b['minute'], df_b['fee_bps'],
         color=EMERALD, lw=1.2, alpha=0.9, label='Dynamic Fee (bps)')
ax2.axhline(BASE_FEE_BPS, color=EMERALD, lw=0.8, ls='--', alpha=0.6,
            label=f'Floor {BASE_FEE_BPS} bps')
ax2.axhline(MAX_FEE_BPS,  color='salmon', lw=0.8, ls='--', alpha=0.7,
            label=f'Ceiling {MAX_FEE_BPS} bps')
ax2.set_ylabel('Dynamic Fee (bps)', color=EMERALD, fontsize=12)
ax2.tick_params(axis='y', labelcolor=EMERALD)
ax2.set_ylim(0, MAX_FEE_BPS * 1.3)

l1, lb1 = ax1.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, lb1 + lb2, loc='lower left', fontsize=9)
plt.title('Fee Elasticity vs. Market Volatility', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()

path = os.path.join(PLOTS_DIR, 'chart1_fee_elasticity.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {path}')

## 6. Chart 2 — Cumulative Fee Revenue Divergence

Both models accrue fees at the same rate during equilibrium, but Model B accelerates sharply through the crash where its elevated fee captures far more revenue from the same toxic arbitrage flow.

In [ ]:
sns.set_style('darkgrid')
fig, ax = plt.subplots(figsize=(14, 6))
_shade(ax)

ax.plot(df_a['minute'], df_a['cum_fees_usdc'],
        color=NAVY,    lw=2.0, label='Model A — Static 0.3%')
ax.plot(df_b['minute'], df_b['cum_fees_usdc'],
        color=EMERALD, lw=2.0, label='Model B — Dynamic Fee')

ax.axvline(480, color=GREY, lw=1.0, ls='--', alpha=0.8)
ax.text(486, df_b['cum_fees_usdc'].iloc[482] * 1.04 + 50,
        'Crash begins', fontsize=8, color=GREY)

ax.set_xlabel('Time (minutes)', fontsize=12)
ax.set_ylabel('Cumulative Fees Captured (USDC)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.legend(fontsize=10)
plt.title('Cumulative Fee Revenue: Static vs. Dynamic Protocol',
          fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()

path = os.path.join(PLOTS_DIR, 'chart2_fee_divergence.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {path}')

## 7. Chart 3 — Net LP Equity & Value Preservation

All three positions start at $600,000. As the pool price diverges from the initial ratio both LPs suffer impermanent loss, but Model B's larger retained fee income lifts its mark-to-market value above Model A's throughout — the EMERALD line stays above NAVY, widening through the crash.

In [ ]:
sns.set_style('darkgrid')
fig, ax = plt.subplots(figsize=(14, 6))
_shade(ax)

ax.plot(df_a['minute'], df_a['hodl_value_usdc'],
        color=GOLD,    lw=1.8, ls='--',
        label='HODL Baseline (initial tokens at market price)')
ax.plot(df_a['minute'], df_a['pool_value_usdc'],
        color=NAVY,    lw=2.0,
        label='Model A — Static-Fee LP')
ax.plot(df_b['minute'], df_b['pool_value_usdc'],
        color=EMERALD, lw=2.0,
        label='Model B — Dynamic-Fee LP')
ax.axhline(INITIAL_VALUE, color='white', lw=0.8, ls=':',
           alpha=0.4, label=f'Initial Value ${INITIAL_VALUE:,.0f}')

ax.set_xlabel('Time (minutes)', fontsize=12)
ax.set_ylabel('Position Value (USDC)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.legend(fontsize=9)
plt.title('Net LP Equity: Dynamic Fee Outperforms Under Stress',
          fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()

path = os.path.join(PLOTS_DIR, 'chart3_lp_equity.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {path}')

## 8. Terminal Summary — Protocol Alpha Capture

In [ ]:
a_fees  = df_a['cum_fees_usdc'].iloc[-1]
a_il    = df_a['il_usdc'].min()
a_net   = df_a['net_lp_return_usdc'].iloc[-1]
a_label = 'Capital Preserved' if a_net >= 0 else 'Capital Impaired'

b_fees  = df_b['cum_fees_usdc'].iloc[-1]
b_il    = df_b['il_usdc'].min()
b_net   = df_b['net_lp_return_usdc'].iloc[-1]
b_label = 'Capital Preserved via Volatility Premium' if b_net >= 0 else 'Capital Impaired'

alpha = b_net - a_net

print('=' * 62)
print('        BACKTEST SIMULATION COMPLETE')
print('=' * 62)
print(f'  Initial Pool Value    : {INITIAL_VALUE:>12,.2f} USDC')
print()
print('  [MODEL A — STATIC 0.3% POOL]')
print(f'  Final Cumulative Fees : +{a_fees:>10,.2f} USDC')
print(f'  Max IL Suffered       :  {a_il:>10,.2f} USDC')
print(f'  Net LP Return         :  {a_net:>+10,.2f} USDC  ({a_label})')
print()
print('  [MODEL B — DYNAMIC AMM PROTOCOL]')
print(f'  Final Cumulative Fees : +{b_fees:>10,.2f} USDC')
print(f'  Max IL Suffered       :  {b_il:>10,.2f} USDC')
print(f'  Net LP Return         :  {b_net:>+10,.2f} USDC  ({b_label})')
print()
print(f'  PROTOCOL ALPHA CAPTURE: +{alpha:>10,.2f} USDC outperformance.')
print('=' * 62)